Overview


In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

file_path = "../data/raw/Analise_LongNeck_WSNP - Sem repostas.xlsb"

xls = pd.ExcelFile(file_path)
df = pd.read_excel(xls, sheet_name="Cenário Divulgado", header=None, engine="pyxlsb")
df_data = df.iloc[:, :]

In [2]:
regioes_validas = ["Mapapi", "NE Norte", "NE Sul", "NO Araguaia", "NO Centro"]

df_reg = df[df.iloc[:, 2].isin(regioes_validas)].copy()

cols_demanda = [3, 16, 27, 38]

demanda_total = pd.DataFrame()

for i, col in enumerate(cols_demanda):
    
    resumo = (
        df_reg
        .assign(valor=pd.to_numeric(df_reg.iloc[:, col], errors="coerce"))
        .groupby(df_reg.iloc[:, 2])["valor"]
        .sum()
    )
    
    demanda_total[f"W{i}"] = resumo

demanda_total.index.name = "Regiao"

demanda_total["Total_Mensal"] = demanda_total.sum(axis=1)
demanda_total.loc["Total_Semanal"] = demanda_total.sum(axis=0)

demanda_total = demanda_total.round(0).astype(int)

In [3]:
regioes_validas = ["Mapapi", "NE Norte", "NE Sul", "NO Araguaia", "NO Centro"]

df_reg = df[df.iloc[:, 2].isin(regioes_validas)].copy()

cols_wsnp = [5, 18, 29, 40]

wsnp_total = pd.DataFrame()

for i, col in enumerate(cols_wsnp):
    
    resumo = (
        df_reg
        .assign(valor=pd.to_numeric(df_reg.iloc[:, col], errors="coerce"))
        .groupby(df_reg.iloc[:, 2])["valor"]
        .sum()
    )
    
    wsnp_total[f"W{i}"] = resumo

wsnp_total.index.name = "Regiao"

wsnp_total["Total_Mensal"] = wsnp_total.sum(axis=1)
wsnp_total.loc["Total_Semanal"] = wsnp_total.sum(axis=0)

wsnp_total = wsnp_total.round(0).astype(int)

In [4]:
demanda_total.style.set_caption("Demanda Total por Região e Semana")

,W0,W1,W2,W3,Total_Mensal
Regiao,,,,,
Mapapi,13553,15041,10701,12332,51627
NE Norte,9450,9693,7394,8123,34660
NE Sul,8948,9390,6993,7754,33086
NO Araguaia,380,459,317,334,1489
NO Centro,7431,8108,6109,6808,28455
Total_Semanal,39762,42691,31514,35351,149318


In [5]:
wsnp_total.style.set_caption("Weekly Supply Network Planning por Região e Semana")

,W0,W1,W2,W3,Total_Mensal
Regiao,,,,,
Mapapi,0,0,0,0,0
NE Norte,27000,14400,23760,12600,77760
NE Sul,0,0,0,0,0
NO Araguaia,0,0,0,0,0
NO Centro,12240,10800,12600,12600,48240
Total_Semanal,39240,25200,36360,25200,126000


In [6]:
df_gap = wsnp_total.sub(demanda_total, fill_value=0)
df_gap.style.set_caption("Gap (Produção - Demanda)")

,W0,W1,W2,W3,Total_Mensal
Regiao,,,,,
Mapapi,-13553,-15041,-10701,-12332,-51627
NE Norte,17550,4707,16366,4477,43100
NE Sul,-8948,-9390,-6993,-7754,-33086
NO Araguaia,-380,-459,-317,-334,-1489
NO Centro,4809,2692,6491,5792,19785
Total_Semanal,-522,-17491,4846,-10151,-23318


Riscos:

In [7]:
# Remover linha agregada que não é região
df_gap_clean = df_gap.drop("Total_Semanal", errors="ignore")

# Calcular necessidade total de transferência
transfer_needed = abs(df_gap_clean[df_gap_clean["Total_Mensal"] < 0]["Total_Mensal"]).sum()

# Calcular superávit local
local_surplus = df_gap_clean[df_gap_clean["Total_Mensal"] > 0]["Total_Mensal"].sum()

# Criar tabela de regiões com déficit
df_transfer_need = df_gap_clean[df_gap_clean["Total_Mensal"] < 0].copy()

# Converter valores negativos em positivos (volume a transferir)
df_transfer_need["transfer_needed"] = abs(df_transfer_need["Total_Mensal"])

df_transfer_need.style.set_caption("Regiões com Necessidade de Transferência")

,W0,W1,W2,W3,Total_Mensal,transfer_needed
Regiao,,,,,,
Mapapi,-13553,-15041,-10701,-12332,-51627,51627
NE Sul,-8948,-9390,-6993,-7754,-33086,33086
NO Araguaia,-380,-459,-317,-334,-1489,1489


In [8]:
local_surplus

np.int64(62885)

In [9]:
transfer_needed

np.int64(86202)

In [10]:
row_total = 25

data = {
    "week": ["W0","W1","W2","W3"],
    "demanda_semanal": [
        df.iloc[row_total,3],
        df.iloc[row_total,16],
        df.iloc[row_total,27],
        df.iloc[row_total,38]
    ],
    "estoque_final": [
        df.iloc[row_total,13],
        df.iloc[row_total,24],
        df.iloc[row_total,35],
        df.iloc[row_total,46]
    ]
}

doi_table = pd.DataFrame(data)

doi_table["demanda_diaria"] = doi_table["demanda_semanal"] / 7
doi_table["DOI"] = doi_table["estoque_final"] / doi_table["demanda_diaria"]

doi_table["risco"] = doi_table["DOI"].apply(
    lambda x: "Crítico" if x < 12 else "Seguro"
)

doi_table.style.set_caption("DOI (Days of Inventory) por Semana da MALZBIER")

,week,demanda_semanal,estoque_final,demanda_diaria,DOI,risco
0,W0,10448.676000,13386.983280,1492.668000,8.968494,Crítico
1,W1,11250.594000,11213.562300,1607.227714,6.976959,Crítico
2,W2,8005.806000,23780.153760,1143.686571,20.792544,Seguro
3,W3,9228.960000,14551.281060,1318.422857,11.036885,Crítico


Observamos que o DOI mínimo exigido é de 12 dias.
As semanas W0 e W1 apresentam níveis abaixo desse limite, indicando risco de ruptura.
Como o lead time de cabotagem é de 25 dias, utilizamos transporte rodoviário emergencial (6 dias) para estabilizar o estoque no curto prazo, enquanto a cabotagem é utilizada para recompor estoques nas semanas seguintes com menor custo.


Logistica


In [11]:
df_demand_coverage = pd.DataFrame({
    "week": ["W0", "W1", "W2", "W3"],
    "demand": demanda_total.loc["Total_Semanal", ["W0","W1","W2","W3"]].values,
    "production": wsnp_total.loc["Total_Semanal", ["W0","W1","W2","W3"]].values
})

# calcular transferência necessária
df_demand_coverage["transfer"] = (
    df_demand_coverage["demand"] - df_demand_coverage["production"]
)

# transferência só existe se houver déficit
df_demand_coverage["transfer"] = df_demand_coverage["transfer"].clip(lower=0)

df_demand_coverage.style.set_caption("Plano de Atendimento da Demanda")

,week,demand,production,transfer
0,W0,39762,39240,522
1,W1,42691,25200,17491
2,W2,31514,36360,0
3,W3,35351,25200,10151


In [12]:
df_transfer_origin = df_transfer_need[["transfer_needed"]].reset_index()

origem_map = {
    "Mapapi": "São Paulo",
    "NE Sul": "São Paulo",
    "NO Araguaia": "Uberlândia"
}

df_transfer_origin["origem"] = df_transfer_origin["Regiao"].map(origem_map)

df_transfer_origin = df_transfer_origin.rename(columns={
    "Regiao": "destino",
    "transfer_needed": "volume"
})

df_transfer_origin = df_transfer_origin[["origem", "destino", "volume"]]

df_transfer_origin.style.set_caption("Plano Logístico de Transferências")

,origem,destino,volume
0,São Paulo,Mapapi,51627
1,São Paulo,NE Sul,33086
2,Uberlândia,NO Araguaia,1489


Custo


In [13]:
df_transferencia = pd.read_excel(xls, sheet_name="Custos de transferência", header=None, engine="pyxlsb")

df_transf = df_transferencia.iloc[3:9, 1:5].copy()
df_transf.columns = ["SKU", "Origem", "Destino", "Reais_por_HL"]
df_transf.reset_index(drop=True, inplace=True)
df_transf["Reais_por_HL"] = df_transf["Reais_por_HL"].astype(float)

df_transf.style.set_caption("Custo de Transferências")

,SKU,Origem,Destino,Reais_por_HL
0,COLORADO LAGER LN 355ML CX C/12,BR16 - F. JACAREI - PLANT - SP,BR04 - F. CAMACARI - PLANT - BA,76.592891
1,COLORADO LAGER LN 355ML CX C/12,BR16 - F. JACAREI - PLANT - SP,BR06 - F. FONTE MATA - RDC - PB,82.079980
2,GOOSE ISLAND MIDWAY NAC LN 355ML CX C/12,BR16 - F. JACAREI - PLANT - SP,BR04 - F. CAMACARI - PLANT - BA,82.395382
3,MALZBIER BRAHMA LN355ML SIX PAC BAND C/4,BR23 - F. JAGUARIUNA - PLANT - SP,BR04 - F. CAMACARI - PLANT - BA,84.579425
4,GOOSE ISLAND MIDWAY NAC LN 355ML CX C/12,BR16 - F. JACAREI - PLANT - SP,BR06 - F. FONTE MATA - RDC - PB,88.298160
5,MALZBIER BRAHMA LN355ML SIX PAC BAND C/4,BR23 - F. JAGUARIUNA - PLANT - SP,BR06 - F. FONTE MATA - RDC - PB,95.328283


In [14]:
df_macos = df_transferencia.iloc[13:16, [1,4]].copy()
df_macos.columns = ["SKU", "Reais_por_HL"]
df_macos.reset_index(drop=True, inplace=True)
df_macos["Reais_por_HL"] = df_macos["Reais_por_HL"].astype(float)

df_macos.style.set_caption("MACOs - Produção Interna")

,SKU,Reais_por_HL
0,COLORADO LAGER LN 355ML CX C/12,300.000000
1,GOOSE ISLAND MIDWAY NAC LN 355ML CX C/12,350.000000
2,MALZBIER BRAHMA LN355ML SIX PAC BAND C/4,285.000000


In [15]:
df_producao = df_transferencia.iloc[20:23, [1,4]].copy()
df_producao.columns = ["SKU", "Reais_por_HL"]
df_producao.reset_index(drop=True, inplace=True)
df_producao["Reais_por_HL"] = df_producao["Reais_por_HL"].astype(float)

df_producao.style.set_caption("Custo de Produção por SKU")

,SKU,Reais_por_HL
0,COLORADO LAGER LN 355ML CX C/12,150.000000
1,GOOSE ISLAND MIDWAY NAC LN 355ML CX C/12,155.000000
2,MALZBIER BRAHMA LN355ML SIX PAC BAND C/4,149.000000


In [16]:
origem_map = {
    "BR16 - F. JACAREI - PLANT - SP": "São Paulo",
    "BR23 - F. JAGUARIUNA - PLANT - SP": "São Paulo"
}

destino_map = {
    "BR04 - F. CAMACARI - PLANT - BA": "NE Sul",
    "BR06 - F. FONTE MATA - RDC - PB": "MAPAPI"
}

df_transf_cost = df_transf.copy()

df_transf_cost["origem"] = df_transf_cost["Origem"].map(origem_map)
df_transf_cost["destino"] = df_transf_cost["Destino"].map(destino_map)

df_transf_cost = df_transf_cost.rename(columns={
    "Reais_por_HL": "custo_HL"
})

df_transfer_cost = df_transfer_origin.merge(
    df_transf_cost,
    on=["origem", "destino"],
    how="left"
)

df_transfer_cost = df_transfer_cost[df_transfer_cost["custo_HL"].notna()]

df_transfer_cost["custo_total"] = (
    df_transfer_cost["volume"] * df_transfer_cost["custo_HL"]
)

df_transfer_cost.style.set_caption("Custo Logístico por Rota")

,origem,destino,volume,SKU,Origem,Destino,custo_HL,custo_total
1,São Paulo,NE Sul,33086,COLORADO LAGER LN 355ML CX C/12,BR16 - F. JACAREI - PLANT - SP,BR04 - F. CAMACARI - PLANT - BA,76.592891,2534152.380947
2,São Paulo,NE Sul,33086,GOOSE ISLAND MIDWAY NAC LN 355ML CX C/12,BR16 - F. JACAREI - PLANT - SP,BR04 - F. CAMACARI - PLANT - BA,82.395382,2726133.621940
3,São Paulo,NE Sul,33086,MALZBIER BRAHMA LN355ML SIX PAC BAND C/4,BR23 - F. JAGUARIUNA - PLANT - SP,BR04 - F. CAMACARI - PLANT - BA,84.579425,2798394.862320


In [17]:
df_modal_cost = df_transf.copy()

df_modal_cost["custo_cabotagem"] = df_modal_cost["Reais_por_HL"]
df_modal_cost["custo_ferrovia"] = df_modal_cost["Reais_por_HL"] * 1.6

df_modal_cost = df_modal_cost[["SKU","custo_cabotagem","custo_ferrovia"]]

df_modal_cost.style.set_caption("Comparação de Custos por Modal (R$/HL)")

,SKU,custo_cabotagem,custo_ferrovia
0,COLORADO LAGER LN 355ML CX C/12,76.592891,122.548625
1,COLORADO LAGER LN 355ML CX C/12,82.079980,131.327968
2,GOOSE ISLAND MIDWAY NAC LN 355ML CX C/12,82.395382,131.832612
3,MALZBIER BRAHMA LN355ML SIX PAC BAND C/4,84.579425,135.327080
4,GOOSE ISLAND MIDWAY NAC LN 355ML CX C/12,88.298160,141.277056
5,MALZBIER BRAHMA LN355ML SIX PAC BAND C/4,95.328283,152.525253


In [18]:
df_prod_cost = df_macos.merge(
    df_producao,
    on="SKU",
    how="inner"
)

df_prod_cost = df_prod_cost.rename(columns={
    "MACO_R$/HL": "maco",
    "custo_producao_R$/HL": "custo_producao"
})

df_prod_cost.style.set_caption("MACO vs Custo de Produção (R$/HL)")

,SKU,Reais_por_HL_x,Reais_por_HL_y
0,COLORADO LAGER LN 355ML CX C/12,300.000000,150.000000
1,GOOSE ISLAND MIDWAY NAC LN 355ML CX C/12,350.000000,155.000000
2,MALZBIER BRAHMA LN355ML SIX PAC BAND C/4,285.000000,149.000000


Produção


In [19]:
df_producao = pd.read_excel(xls, sheet_name="Produção PCP", header=None, engine="pyxlsb")
df_br03 = df_producao.iloc[2:5].copy() 
df_br03.columns = [
    "Location",
    "Linha",
    "Nominal_grf_h",
    "Capacity_hl",
    "SKU",
    "Container",
    "02_02_2026",
    "09_02_2026",
    "16_02_2026",
    "23_02_2026"
]

df_br03[["Location","Linha","Nominal_grf_h","Capacity_hl"]] = \
df_br03[["Location","Linha","Nominal_grf_h","Capacity_hl"]].ffill()


In [20]:
df_br31 = df_producao.iloc[9:15].copy()

df_br31.columns = df_br03.columns

df_br31[["Location","Linha","Nominal_grf_h","Capacity_hl"]] = \
df_br31[["Location","Linha","Nominal_grf_h","Capacity_hl"]].ffill()


In [21]:
df_pcp = pd.concat([df_br03, df_br31], ignore_index=True)

df_pcp.style.set_caption("Plano Mestre de Produção (PCP) - BR03 e BR31")

,Location,Linha,Nominal_grf_h,Capacity_hl,SKU,Container,02_02_2026,09_02_2026,16_02_2026,23_02_2026
0,BR03 - F. AQUIRAZ - PLANT - CE,L541,72000,12600,MALZBIER BRAHMA LN355ML SIX PAC BAND C/4,LONG NECK 355ML,0,9000.000000,7560.000000,0.000000
1,BR03 - F. AQUIRAZ - PLANT - CE,L541,72000,12600,PATAGONIA AMBER LAGER LN355ML CX12,LONG NECK 355ML,12240,1800.000000,5040.000000,12600.000000
2,BR03 - F. AQUIRAZ - PLANT - CE,L541,72000,12600,COLORADO LAGER LN 355ML CX C/12,LONG NECK 355ML,0,0.000000,0.000000,0.000000
3,BR31 - F. PERNAMBUCO - PLANT - PE,L541,108000,27000,BRAHMA CHOPP ZERO LN 355 SIXP CX CART C4,LONG NECK 355ML,0,0.000000,0.000000,3600.000000
4,BR31 - F. PERNAMBUCO - PLANT - PE,L541,108000,27000,GOOSE ISLAND MIDWAY NAC LN 355ML CX C/12,LONG NECK 330ML,5400,14400.000000,0.000000,12600.000000
5,BR31 - F. PERNAMBUCO - PLANT - PE,L541,108000,27000,MALZBIER BRAHMA LN355ML SIX PAC BAND C/4,LONG NECK 330ML,16200,0.000000,12960.000000,0.000000
6,BR31 - F. PERNAMBUCO - PLANT - PE,L541,108000,27000,COLORADO LAGER LN 355ML CX C/12,LONG NECK 355ML,5400,0.000000,10800.000000,0.000000
7,BR31 - F. PERNAMBUCO - PLANT - PE,L541,108000,27000,SKOL BEATS SENSES LN 269ML SIXPACK SH C4,LONG NECK 269ML,0,0.000000,3240.000000,0.000000
8,BR31 - F. PERNAMBUCO - PLANT - PE,L541,108000,27000,BUDWEISER ZERO LN 330ML SIX-PACK SH C/4,LONG NECK 330ML,0,5400.000000,0.000000,10800.000000


In [22]:
cols_semana = [
    "02_02_2026",
    "09_02_2026",
    "16_02_2026",
    "23_02_2026"
]

df_pcp["producao_total"] = df_pcp[cols_semana].sum(axis=1)

df_capacity = df_pcp.groupby("Location").agg({
    "Capacity_hl": "first",
    "producao_total": "sum"
}).reset_index()

df_capacity["utilizacao_%"] = (
    df_capacity["producao_total"] / (df_capacity["Capacity_hl"] *4)
) * 100

In [23]:
df_capacity = df_capacity.rename(columns={
    "Location": "planta",
    "Capacity_hl": "capacidade_hl"
})

df_capacity.style.set_caption("Utilização da Capacidade Produtiva")

,planta,capacidade_hl,producao_total,utilizacao_%
0,BR03 - F. AQUIRAZ - PLANT - CE,12600,48240.000000,95.714286
1,BR31 - F. PERNAMBUCO - PLANT - PE,27000,100800.000000,93.333333


In [24]:
demanda_total_kpi = demanda_total.loc["Total_Semanal"].sum()

demanda_total_kpi

np.int64(298636)

In [25]:
producao_total_kpi = wsnp_total.loc["Total_Semanal"].sum()

producao_total_kpi

np.int64(252000)

In [26]:
deficit_total_kpi = abs(df_gap.loc["Total_Semanal"].sum())

deficit_total_kpi

np.int64(46636)

In [27]:
custo_total_kpi = df_transfer_cost["custo_total"].sum()

custo_total_kpi

np.float64(8058680.865206869)